![Imgur](https://i.imgur.com/5pXzCIu.png)

# Data Science va Sun'iy Intellekt Praktikum

## 5-MODUL. Machine Learning

### Portfolio uchun vazifa: Toshkent shahrida uylarning narxini aniqlash.

Ushbu amaliyotda sizning vazifangiz berilgan ma`lumotlar asosida Toshkent shahridagi uylarning narxini aniqlash.

In [15]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
df = pd.read_csv('https://raw.githubusercontent.com/anvarnarz/praktikum_datasets/main/housing_data_08-02-2021.csv')
df.head()

,location,district,rooms,size,level,max_levels,price
0,"город Ташкент, Юнусабадский район, Юнусабад 8-...",Юнусабадский,3,57,4,4,52000
1,"город Ташкент, Яккасарайский район, 1-й тупик ...",Яккасарайский,2,52,4,5,56000
2,"город Ташкент, Чиланзарский район, Чиланзар 2-...",Чиланзарский,2,42,4,4,37000
3,"город Ташкент, Чиланзарский район, Чиланзар 9-...",Чиланзарский,3,65,1,4,49500
4,"город Ташкент, Чиланзарский район, площадь Актепа",Чиланзарский,3,70,3,5,55000


# Ustunlar ta'rifi
- `location` - sotilayotgan uy manzili
- `district` - uy joylashgan tuman
- `rooms` - xonalar soni
- `size` - uy maydoni (kv.m)
- `level` - uy joylashgan qavat
- `max_levels` - ja'mi qavatlar soni
- `price` - uy narxi

## Vazifani CRSIP-DM Metolodgiyasi yordamida bajaring.
<img src="https://i.imgur.com/dzZnnYi.png" alt="CRISP-DM" width="800"/>

Data Preparation

In [16]:
df.loc[df[df['size'] == 'Площадьземли:1сот'].index, 'size'] = np.nan # changing type of some columns and fill null values
df.loc[df[df['price'] == 'Договорная'].index, 'price'] = np.nan
df[['size','price']] = df[['size','price']].astype(float)
df.fillna(df.mean(numeric_only = True), inplace = True)

df.drop(df[df['size'] >= 500].index, inplace = True)   # working with outliers
idx  = df[(df.price > 300000) & (df.rooms < 7) & (df['size'] < 200)].index
df.drop(idx, inplace= True)
idx2 = df[df.price > 400000].index
df.drop(idx2, inplace = True)

df['size_per_room'] = df['size'] / df['rooms'] # feature engineering

df.drop('location', axis = 1, inplace = True) # drop 'location' column

Train and Test set

In [17]:
from sklearn.model_selection import train_test_split
train_set, test_set = train_test_split(df, test_size = 0.2, random_state = 42)

housing = train_set.drop('price', axis = 1)
housing_num = housing.drop('district', axis = 1)
y = train_set['price']

In [18]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder

num_attribs = list(housing_num.columns)
cat_attribs = ['district']

full_pipeline = ColumnTransformer([
    ('num', StandardScaler(), num_attribs),
    ('cat', OneHotEncoder(), cat_attribs)
])

housing_prepared = full_pipeline.fit_transform(housing)
housing_prepared[:5]

array([[-1.51381303, -1.36066253, -1.20686316, -0.78536517, -0.59114316,
         0.        ,  0.        ,  1.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ],
       [-0.58296194,  0.29694879, -0.31697264, -0.78536517,  1.67833315,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  1.        ,  0.        ,  0.        ,
         0.        ,  0.        ],
       [-0.58296194, -1.11202083,  1.90775365,  1.13229801, -1.53675829,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         1.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ],
       [-0.58296194, -0.697618  , -0.31697264, -0.40183253, -0.59114316,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  1.        ,  0.        ,  0.        ,
         0.        

Build Regression Model

In [19]:
from sklearn.ensemble import RandomForestRegressor

RF_model = RandomForestRegressor()

RF_model.fit(housing_prepared, y)

RandomForestRegressor()

Testing

In [20]:
X_test = test_set.drop('price', axis = 1)
y_test = test_set['price']

X_test_prepared = full_pipeline.transform(X_test)

y_predicted = RF_model.predict(X_test_prepared)

from sklearn.metrics import root_mean_squared_error, mean_squared_error
rmse = root_mean_squared_error(y_test, y_predicted)
rmse

17487.74899182588

In [21]:
from sklearn.metrics import mean_absolute_error
mae = mean_absolute_error(y_test, y_predicted)
mae

9536.121489072957

Cross Val Score


In [22]:
X = df.drop('price', axis = 1)
y_cros = df['price'].copy()
X_prepared = full_pipeline.transform(X)

from sklearn.model_selection import cross_val_score
rmse_scores = cross_val_score(RF_model, X_prepared, y_cros, scoring = 'neg_mean_squared_error', cv = 10)

rmse  = np.sqrt(-rmse_scores)

print(f"""RMSE: {rmse}
      Mean:{rmse.mean()}
      STD: {rmse.std()}""")

RMSE: [15425.79702053 16812.98322469 25208.94853402 24330.15688358
 20832.04641616 23938.88474249 20324.07908861 19139.71840974
 18440.92876404 15739.20965876]
      Mean:20019.275274262087
      STD: 3385.223187653777


Saving model

In [23]:
import joblib
filename = 'RF_model'
joblib.dump(RF_model, filename)

['RF_model']

Loading model

In [24]:
model = joblib.load(filename)